<a href="https://colab.research.google.com/github/DachsteinSilalahi/Final-project-hacktiv8/blob/main/Final_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q streamlit pyngrok google-genai

In [ ]:
%pip install -qU langgraph langchain-google-genai

In [ ]:
from pyngrok import ngrok
from google.colab import userdata

ngrok.set_auth_token(userdata.get('NGROK_TOKEN'))
print("ngrok token berhasil digunakan!")

ngrok token berhasil digunakan!


In [ ]:
import os
import google.generativeai as genai

GOOGLE_API_KEY = userdata.get('GEMINI')
os.environ['GOOGLE_API_KEY'] = GOOGLE_API_KEY

In [ ]:
import subprocess
import time
from pyngrok import ngrok

def run_streamlit(filename, port=8501):
    subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
    subprocess.run(["fuser", "-k", f"{port}/tcp"], capture_output=True)
    subprocess.run(["killall", "-9", "ngrok"], capture_output=True)
    ngrok.kill()
    time.sleep(5)

    print(f"Starting Streamlit app: {filename} on port {port}")
    proc = subprocess.Popen(
        [
            "streamlit", "run", filename,
            "--server.headless=true",
            "--server.port", str(port),
            "--server.enableCORS=false",
        ],
        stdout=None,
        stderr=None
    )

    time.sleep(10)

    try:
        public_url = ngrok.connect(port)
        print(f"Streamlit berjalan: {public_url}")
    except Exception as e:
        print(f"Gagal menghubungkan ngrok: {e}")
        print("Coba jalankan kembali cell ini. Jika masih gagal, coba restart runtime Colab Anda.")
        public_url = None

    return proc

In [ ]:
%%writefile plan_app.py
import streamlit as st
import pandas as pd
import google.generativeai as genai

SYSTEM_INSTRUCTION = """
Anda adalah 'FinPlan AI Expert'. Tugas utama Anda:
1. *Mencatat Pengeluaran*: Jika user menyebutkan nominal belanja, simpan informasi tersebut.
2. *Menghitung Budget*: Gunakan rumus 50/30/20 (Kebutuhan/Keinginan/Tabungan) kecuali user meminta yang lain.
3. *Tips Menabung & Penghematan*: Berikan rekomendasi konkret, misal: 'Kurangi membeli barang yang tidak diperlukan'.
4. *Target Keuangan*: Bantu user menghitung berapa lama target tercapai (Misal: Ingin beli motor 25jt, tabungan 5jt/bulan = 5 bulan).

Gaya bahasa: Informatif, menggunakan poin-poin (bullet points), dan profesional.
"""

st.title("Financial Plan AI")
st.write("""
## Selamat datang sayangnya aku!

Saya adalah chatbot kesayangan mu yang akan membantu kamu dalam merencakan keuangan kamu
""")

with st.sidebar:
    st.header("Ringkasan Keuangan")
    st.info("Gunakan aku untuk mencatat pengeluaran kamu secara otomatis.")

    st.subheader("Target Kamu")
    target_name = st.text_input("Nama Target", "Dana Darurat")
    target_amount = st.number_input("Nominal (Rp)", value=1000000000)

    st.subheader("Kalkulator Cepat 50/30/20")
    gaji = st.number_input("Input Gaji Bulanan", value=0)
    if gaji > 0:
        st.write(f"Kebutuhan (50%): Rp {gaji*0.5:,.0f}")
        st.write(f"Keinginan (30%): Rp {gaji*0.3:,.0f}")
        st.write(f"Tabungan (20%): Rp {gaji*0.2:,.0f}")

st.title("Financial Plan AI: Rencakan Keuangan Kamu ")

if "chat_session" not in st.session_state:
    model = genai.GenerativeModel(
        model_name="gemini-2.5-flash",
        system_instruction=SYSTEM_INSTRUCTION
    )
    st.session_state.chat_session = model.start_chat(history=[])

for message in st.session_state.chat_session.history:
    with st.chat_message("user" if message.role == "user" else "assistant"):
        st.markdown(message.parts[0].text)

if prompt := st.chat_input("Contoh: 'Catat pengeluaran makan siang 50rb' atau 'Beri tips hemat listrik'"):
    st.chat_message("user").markdown(prompt)

    full_prompt = f"{prompt}. Jika ini pengeluaran, berikan rekomendasi penghematan terkait hal tersebut."

    response = st.session_state.chat_session.send_message(full_prompt)
    with st.chat_message("assistant"):
        st.markdown(response.text)

Overwriting plan_app.py


In [ ]:
proc = run_streamlit("plan_app.py")

Starting Streamlit app: plan_app.py on port 8501
Streamlit berjalan: NgrokTunnel: "https://clergyman-guy-splinter.ngrok-free.dev" -> "http://localhost:8501"


In [81]:
try:
    proc.terminate()
    print("Streamlit dihentikan.")
except:
    print("Tidak ada proses yang berjalan.")

ngrok.kill()
print("Tunnel ngrok ditutup.")

Streamlit dihentikan.
Tunnel ngrok ditutup.
